In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import re

train_df = pd.read_parquet('data/raw/train.parquet')
benchmark_queries_df = pd.read_parquet('data/raw/benchmark_queries.parquet')
benchmark_items_df = pd.read_parquet('data/raw/benchmark_items.parquet')

# Создаем копии для работы
train_fe = train_df.copy()
queries_fe = benchmark_queries_df.copy()
items_fe = benchmark_items_df.copy()

print("Данные загружены для feature engineering")

Данные загружены для feature engineering


In [2]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Нормализация текста
    text = str(text).lower()
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Очистка текстовых признаков...")

# Для train
train_fe['search_query_clean'] = train_fe['search_query'].apply(clean_text)
train_fe['item_title_clean'] = train_fe['item_title_raw'].apply(clean_text)
train_fe['item_description_clean'] = train_fe['item_description_raw'].apply(clean_text)

# Для benchmark items
items_fe['item_title_clean'] = items_fe['item_title_raw'].apply(clean_text)
items_fe['item_description_clean'] = items_fe['item_description_raw'].apply(clean_text)

# Для benchmark queries
queries_fe['search_query_clean'] = queries_fe['search_query'].apply(clean_text)

print("Тексты очищены")

Очистка текстовых признаков...
Тексты очищены


In [3]:
# Для объявлений: объединение заголовков и описаний
items_fe['item_text_combined'] = (
    items_fe['item_title_clean'] + ' ' + 
    items_fe['item_description_clean'].fillna('')
).str.strip()

# Для запросов: только поисковый запрос
# (search_infm_params_text можно добавить позже как отдельный признак)
queries_fe['query_text_combined'] = queries_fe['search_query_clean']


train_fe['item_text_combined'] = (
    train_fe['item_title_clean'] + ' ' + 
    train_fe['item_description_clean'].fillna('')
).str.strip()

print("Созданы комбинированные текстовые признаки")
print(f"\nПример объединенного текста объявления:")
print(items_fe['item_text_combined'].iloc[0][:200])

Созданы комбинированные текстовые признаки

Пример объединенного текста объявления:
ремонт выкуп компьют и ноутбуков с выездом на дом выезд на дом в любое время вплоть до 23 00 диагностика устройств чистка от пыли и замена термопасты проверка работоспособности всех компонентов ремонт


In [4]:
# Длина текстов
items_fe['title_length'] = items_fe['item_title_clean'].str.len()
items_fe['description_length'] = items_fe['item_description_clean'].str.len()
items_fe['text_combined_length'] = items_fe['item_text_combined'].str.len()

# Число слов
items_fe['title_word_count'] = items_fe['item_title_clean'].str.split().apply(len)
items_fe['description_word_count'] = items_fe['item_description_clean'].str.split().apply(len)
items_fe['text_combined_word_count'] = items_fe['item_text_combined'].str.split().apply(len)

# Число уникальных слов
items_fe['title_unique_words'] = items_fe['item_title_clean'].apply(
    lambda x: len(set(str(x).split()))
)

# Для запросов
queries_fe['query_length'] = queries_fe['search_query_clean'].str.len()
queries_fe['query_word_count'] = queries_fe['search_query_clean'].str.split().apply(len)

print("Созданы статистические текстовые признаки")
print(f"\nСтатистика:")
print(items_fe[['title_length', 'description_length', 'text_combined_length']].describe())

Созданы статистические текстовые признаки

Статистика:
        title_length  description_length  text_combined_length
count  189212.000000       189212.000000         189212.000000
mean       32.962513         1302.186061           1336.148236
std        11.772321         1191.064362           1194.798056
min         1.000000            0.000000              3.000000
25%        24.000000          394.000000            424.000000
50%        34.000000          975.000000           1009.000000
75%        43.000000         1828.000000           1864.000000
max       100.000000         7923.000000           7972.000000


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF векторизатор
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    dtype=np.float32
)

# Обучение на текстах объявлений
print("Обучение TF-IDF векторизатора...")
item_tfidf_matrix = tfidf_vectorizer.fit_transform(items_fe['item_text_combined'])

print(f"TF-IDF матрица создана: {item_tfidf_matrix.shape}")
print(f"   Строк (объявлений): {item_tfidf_matrix.shape[0]}")
print(f"   Столбцов (признаков): {item_tfidf_matrix.shape[1]}")

import joblib
joblib.dump(tfidf_vectorizer, 'data/processed/tfidf_vectorizer.pkl')
print("Векторизатор сохранен")

Обучение TF-IDF векторизатора...
TF-IDF матрица создана: (189212, 50000)
   Строк (объявлений): 189212
   Столбцов (признаков): 50000
Векторизатор сохранен


In [14]:
# Уникальные значения
print("Уникальные категории поиска:")
print(train_fe['search_category'].value_counts())
print(f"\nВсего уникальных категорий: {train_fe['search_category'].nunique()}")

# Кодировка категорий
category_encoder = LabelEncoder()

# Обработка пропусков
train_fe['search_category_filled'] = train_fe['search_category'].fillna('unknown')
queries_fe['search_category_filled'] = queries_fe['search_category'].fillna('unknown')

# Кодировка
train_fe['search_category_encoded'] = category_encoder.fit_transform(
    train_fe['search_category_filled']
)
queries_fe['search_category_encoded'] = category_encoder.transform(
    queries_fe['search_category_filled']
)

print(f"\nКатегории поиска закодированы")
print(f"Пример маппинга: {dict(list(zip(category_encoder.classes_[:5], 
                                      category_encoder.transform(category_encoder.classes_[:5]))))}")

Уникальные категории поиска:
search_category
114    497635
0          34
81          2
10          2
Name: count, dtype: int64

Всего уникальных категорий: 4

Категории поиска закодированы
Пример маппинга: {np.int64(0): np.int64(0), np.int64(10): np.int64(1), np.int64(81): np.int64(2), np.int64(114): np.int64(3)}


In [15]:
# Проверка item_category_id
print("Уникальные категории объявлений (топ-20):")
print(items_fe['item_category_id'].value_counts().head(20))
print(f"\nВсего уникальных категорий: {items_fe['item_category_id'].nunique()}")

# Проверка на пропуски
cat_missing = items_fe['item_category_id'].isnull().sum()
print(f"Объявлений без категории: {cat_missing} ({cat_missing/len(items_fe)*100:.1f}%)")

# Заполнение пропусков
items_fe['item_category_id_filled'] = items_fe['item_category_id'].fillna(-1)

# Кодировка
item_category_encoder = LabelEncoder()
items_fe['item_category_encoded'] = item_category_encoder.fit_transform(
    items_fe['item_category_id_filled'].astype(str)
)

print(f"\nКатегории объявлений закодированы")

Уникальные категории объявлений (топ-20):
item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56
97         44
21         36
88         34
82         32
42         29
Name: count, dtype: int64

Всего уникальных категорий: 47
Объявлений без категории: 0 (0.0%)

Категории объявлений закодированы


In [20]:
# Пропуски
search_loc_missing = train_fe['search_location_id'].isnull().sum()
item_loc_missing = items_fe['item_location_id'].isnull().sum()

print(f"\nЗапросов без локации: {search_loc_missing} ({search_loc_missing/len(train_fe)*100:.1f}%)")
print(f"Объявлений без локации: {item_loc_missing} ({item_loc_missing/len(items_fe)*100:.1f}%)")

# Заполнение пропусков
train_fe['search_location_filled'] = train_fe['search_location_id'].fillna(-1)
items_fe['item_location_filled'] = items_fe['item_location_id'].fillna(-1)
queries_fe['search_location_filled'] = queries_fe['search_location_id'].fillna(-1)


# Кодировка
location_encoder = LabelEncoder()

# Объединенеи лок для согласования кодировки
all_locations = pd.concat([
    train_fe['search_location_filled'],
    items_fe['item_location_filled'],
    queries_fe['search_location_filled']
]).astype(str).unique()

location_encoder.fit(all_locations)

train_fe['search_location_encoded'] = location_encoder.transform(
    train_fe['search_location_filled'].astype(str)
)
items_fe['item_location_encoded'] = location_encoder.transform(
    items_fe['item_location_filled'].astype(str)
)
queries_fe['search_location_encoded'] = location_encoder.transform(
    queries_fe['search_location_filled'].astype(str)
)

print(f"\nЛокации закодированы")


Запросов без локации: 0 (0.0%)
Объявлений без локации: 0 (0.0%)

Локации закодированы


In [21]:
# Создание признака совпадения локации (для train)
train_fe['location_match'] = (
    train_fe['search_location_id'] == train_fe['item_location_id']
).astype(int)

print(f"Создан признак совпадения локации")
print(f"Доля совпадений локаций в train: {train_fe['location_match'].mean()*100:.1f}%")

Создан признак совпадения локации
Доля совпадений локаций в train: 83.1%


In [22]:
# Создание признака наличия координат
items_fe['has_coords'] = (
    items_fe['item_latitude'].notnull() & items_fe['item_longitude'].notnull()
).astype(int)

print(f"\nКоординаты обработаны")
print(f"Объявлений с координатами: {items_fe['has_coords'].sum()}")


Координаты обработаны
Объявлений с координатами: 189211


In [32]:
# Приведение к float64
print("\nПриведение к float64...")
items_fe['item_price'] = pd.to_numeric(items_fe['item_price'], errors='coerce').astype('float64')
print(f"Новый тип: {items_fe['item_price'].dtype}")

# Создание ценовых категорий
items_fe['price_category'] = pd.qcut(
    items_fe['item_price_filled'], 
    q=5, 
    labels=['very_low', 'low', 'medium', 'high', 'very_high'],
    duplicates='drop'
)

# Создание признака "цена равна 0 или отсутствует"
items_fe['price_is_zero_or_missing'] = (
    (items_fe['item_price_filled'] == 0) | 
    (items_fe['item_price'].isnull())
).astype(int)

print(f"\nЦена обработана")
print(f"\nРаспределение ценовых категорий:")
print(items_fe['price_category'].value_counts())
print(f"Доля объявлений с нулевой/отсутствующей ценой: {items_fe['price_is_zero_or_missing'].mean()*100:.1f}%")


Приведение к float64...
Новый тип: float64

Цена обработана

Распределение ценовых категорий:
price_category
high         39077
very_low     38594
medium       38584
low          37115
very_high    35842
Name: count, dtype: int64
Доля объявлений с нулевой/отсутствующей ценой: 0.7%


In [33]:
# Пропуски
rating_missing = items_fe['item_rating'].isnull().sum()
print(f"\nОбъявлений без рейтинга: {rating_missing} ({rating_missing/len(items_fe)*100:.1f}%)")

# Заполнение пропусков средним
items_fe['item_rating_filled'] = items_fe['item_rating'].fillna(
    items_fe['item_rating'].mean()
)

# Создание категориальных признаков рейтинга
items_fe['rating_category'] = pd.cut(
    items_fe['item_rating_filled'],
    bins=[0, 3, 4, 4.5, 5],
    labels=['low', 'medium', 'high', 'very_high']
)

print(f"\nРейтинг обработан")


Объявлений без рейтинга: 17631 (9.3%)

Рейтинг обработан


In [34]:
# Пропуски
reviews_missing = items_fe['item_rating_reviews_count'].isnull().sum()
print(f"\nОбъявлений без отзывов: {reviews_missing} ({reviews_missing/len(items_fe)*100:.1f}%)")

# Заполнение пропусков нулем
items_fe['item_rating_reviews_count_filled'] = items_fe['item_rating_reviews_count'].fillna(0)

# Логарифмическое преобразование
items_fe['reviews_count_log'] = np.log1p(items_fe['item_rating_reviews_count_filled'])

print(f"\nОтзывовы обработаны")


Объявлений без отзывов: 11615 (6.1%)

Отзывовы обработаны


In [35]:
# item_is_phone_hidden
print("Распределение item_is_phone_hidden:")
print(items_fe['item_is_phone_hidden'].value_counts())

# Заполнение пропусков
items_fe['item_is_phone_hidden_filled'] = items_fe['item_is_phone_hidden'].fillna(0)

# item_is_message_forbidden
print("\nРаспределение item_is_message_forbidden:")
print(items_fe['item_is_message_forbidden'].value_counts())

items_fe['item_is_message_forbidden_filled'] = items_fe['item_is_message_forbidden'].fillna(0)

print(f"\nФлаги обработаны")

Распределение item_is_phone_hidden:
item_is_phone_hidden
False    159705
True      29507
Name: count, dtype: int64

Распределение item_is_message_forbidden:
item_is_message_forbidden
False    184179
True       5033
Name: count, dtype: int64

Флаги обработаны


In [36]:
# Проверка фильтров
print("Примеры поисковых фильтров:")
filters_examples = train_fe[train_fe['search_infm_params_text'].notnull()]['search_infm_params_text'].unique()
for i, f in enumerate(filters_examples[:10]):
    print(f"  {i+1}. {f}")

# Создание признаков из фильтров

def extract_filter_features(text):
    if pd.isna(text):
        return {
            'has_filter': 0,
            'filter_length': 0,
            'filter_word_count': 0
        }
    
    text = str(text).lower()
    return {
        'has_filter': 1,
        'filter_length': len(text),
        'filter_word_count': len(text.split())
    }

# Применяем к train
filter_features_train = train_fe['search_infm_params_text'].apply(extract_filter_features)
train_fe['has_filter'] = filter_features_train.apply(lambda x: x['has_filter'])
train_fe['filter_length'] = filter_features_train.apply(lambda x: x['filter_length'])
train_fe['filter_word_count'] = filter_features_train.apply(lambda x: x['filter_word_count'])

# Применяем к queries
filter_features_queries = queries_fe['search_infm_params_text'].apply(extract_filter_features)
queries_fe['has_filter'] = filter_features_queries.apply(lambda x: x['has_filter'])
queries_fe['filter_length'] = filter_features_queries.apply(lambda x: x['filter_length'])
queries_fe['filter_word_count'] = filter_features_queries.apply(lambda x: x['filter_word_count'])

print(f"\nПризнаки фильтров созданы")
print(f"Запросов с фильтрами в train: {train_fe['has_filter'].sum()}")
print(f"Запросов с фильтрами в benchmark: {queries_fe['has_filter'].sum()}")

Примеры поисковых фильтров:
  1. 
  2. Рейтинг пользователя 4 звезды и выше
  3. Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье
  4. Вид услуги Оборудование, производство
  5. Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка
  6. Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отделка
  7. Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье
  8. Вид услуги
  9. Где вы оказываете услуги У себя дома Ваши клиенты Мужчины Кто оказывает услуги Женщина Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье
  10. Вид услуги Автосервис, аренда Тип услуги Автосервис

Признаки фильтров созданы
Запросов с фильтрами в train: 497673
Запросов с фильтрами в benchmark: 2452


In [37]:
# Сохранение
train_fe.to_parquet('data/processed/train_fe.parquet', index=False)
queries_fe.to_parquet('data/processed/queries_fe.parquet', index=False)
items_fe.to_parquet('data/processed/items_fe.parquet', index=False)

from scipy.sparse import save_npz
save_npz('data/processed/item_tfidf_matrix.npz', item_tfidf_matrix)

print("Все обработанные данные сохранены")
print(f"\nРазмеры данных:")
print(f"  train_fe: {train_fe.shape}")
print(f"  queries_fe: {queries_fe.shape}")
print(f"  items_fe: {items_fe.shape}")
print(f"  item_tfidf_matrix: {item_tfidf_matrix.shape}")

Все обработанные данные сохранены

Размеры данных:
  train_fe: (497673, 31)
  queries_fe: (2452, 17)
  items_fe: (189212, 39)
  item_tfidf_matrix: (189212, 50000)
